# Create embeddings of documents

#### Setup Environment

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd
from semantic_text_splitter import TextSplitter
from tokenizers import Tokenizer
import pickle

In [2]:
# Loads variables from the environment
load_dotenv()
open_api_key = os.getenv("OPENAI_API_KEY")

#### Read in and Chunk Documents 

#### Create Embedding Vectors for Chunks

> ToDo: add some sort of loop through all videos

In [3]:
# Load title dictionary from pickle file
with open('data/title_dict.pkl', 'rb') as f:
    title_dict = pickle.load(f)

title_df = pd.DataFrame(title_dict.items(), columns=['video_id', 'title'])
title_df.head()


,video_id,title
0,q-wRvsiGYIs,"AMA #19: Collagen vs. Whey Protein, Creatine, ..."
1,ssmwxKPFMFU,Protocols to Improve Vision & Eyesight | Huber...
2,J7yn4tJEmJU,Tools for Overcoming Substance & Behavioral Ad...
3,7MEhDlw1e9k,How to Build Endurance | Huberman Lab Essentials
4,UyneMnERmnI,How to Improve Your Vitality & Heal From Disea...


In [4]:
client = OpenAI(api_key=open_api_key)

max_tokens = 1023 # 8191 is max length for text-embedding-3-large
tokenizer = Tokenizer.from_pretrained("bert-base-uncased")
splitter = TextSplitter.from_huggingface_tokenizer(tokenizer, max_tokens)

# Create lists to store the data
chunk_embeddings = []
title_embeddings = []
video_ids = []
chunk_ids = []


# Iterates through a list of video IDs, where each ID corresponds to a podcast episode transcript.
# Each transcript will be split into chunks and embedded along with its title for semantic search.
    # Currently using a test subset of videos but will be expanded to process the full dataset.
    # ToDo: add some sort of loop through all videos
documents = ['E7W4OQfJWdw', 'cS7cNaBrkxo']
for video_id in documents:

    # Create Title Embeddings
    title_vector = client.embeddings.create(
        input=title_dict[video_id],
        model="text-embedding-3-small"
    )

    # Load document and split into semantic chunks
    with open(f'data/documents/{video_id}.txt', 'r', encoding='utf-8') as file:
        text_content = file.read()
    # ToDo: validate or rework so chunks are topical sentiments with varying lengths
    chunks = splitter.chunks(text_content) 

    # Create data record of embeddings for each chunk of the transcript
    for chunk in chunks:
        # Create chunks directory if it doesn't exist 
        os.makedirs('data/chunks', exist_ok=True)
        
        # Write chunk to file with video ID and chunk number
        chunk_filename = f'data/chunks/{video_id}_chunk{chunks.index(chunk)}.txt'
        with open(chunk_filename, 'w', encoding='utf-8') as f:
            f.write(chunk)
        
        # Create Chunk Embeddings
        chunk_vector = client.embeddings.create(
            input=chunk,
            model="text-embedding-3-small"
        )
        # Append values of this record to column list
        chunk_embeddings.append(chunk_vector.data[0].embedding)
        title_embeddings.append(title_vector.data[0].embedding)
        video_ids.append(video_id)
        chunk_ids.append(chunks.index(chunk))


/Users/dom/Repos/huberman-lab-rag/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Create DataFrame
df = pd.DataFrame({
    'content_vector': chunk_embeddings,
    'title_vector': title_embeddings,
    'video_id': video_ids, 
    'chunk_id': chunk_ids,
})

In [6]:
df.reset_index(inplace=True)
df.rename(columns={'index': 'vector_id'}, inplace=True)
df.head()

,vector_id,content_vector,title_vector,video_id,chunk_id
0,0,"[0.010170252993702888, -0.015520867891609669, ...","[-0.027921706438064575, 0.025697391480207443, ...",E7W4OQfJWdw,0
1,1,"[0.00484744505956769, 0.008455636911094189, -0...","[-0.027921706438064575, 0.025697391480207443, ...",E7W4OQfJWdw,1
2,2,"[0.017064958810806274, -0.0008320452179759741,...","[-0.027921706438064575, 0.025697391480207443, ...",E7W4OQfJWdw,2
3,3,"[0.018084809184074402, -0.010019644163548946, ...","[-0.027921706438064575, 0.025697391480207443, ...",E7W4OQfJWdw,3
4,4,"[0.046452846378088, -0.005249288398772478, -0....","[-0.027921706438064575, 0.025697391480207443, ...",E7W4OQfJWdw,4


In [7]:
# Save DataFrame to CSV
df.to_csv('data/embeddings.csv', index=False)